# 🧊 Imagen → 3D con TripoSR (gratis, GPU de Colab)

Reconstrucción 3D de una imagen en la **GPU gratuita de Google Colab**. No gasta créditos.

## Pasos
1. **GPU**: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **T4 GPU**.
2. Celda 1: instalar. 3. Celda 2: subir imagen (PNG con fondo transparente = mejor). 4. Celda 3: generar. 5. Celda 4: descargar.

> Animá el `.glb` en https://imagen-a-3d-triposr.vercel.app → “Animar un modelo 3D”.

In [ ]:
# Celda 1 — Instalar TripoSR
!nvidia-smi -L
import os
os.chdir('/content')
if not os.path.isdir('/content/TripoSR'):
    !git clone https://github.com/VAST-AI-Research/TripoSR.git
os.chdir('/content/TripoSR')
!pip install -q --upgrade pip setuptools wheel ninja
!pip install -q 'git+https://github.com/tatsy/torchmcubes.git'
!pip install -q -r requirements.txt
!pip install -q onnxruntime
import torch, torchmcubes
print('torch:', torch.__version__, '| GPU:', torch.cuda.is_available(), '| torchmcubes OK')

In [ ]:
# Celda 2 — Subir imagen (define IMG)
from google.colab import files
from PIL import Image
import os
os.chdir('/content/TripoSR')
up = files.upload()
IMG = list(up.keys())[0]
im = Image.open(IMG)
print('IMG:', IMG, '| modo:', im.mode, '| tamaño:', im.size)

In [ ]:
# Celda 3 — Reconstruir (parche rembg + --no-remove-bg para evitar dependencias rotas)
import os, time
os.chdir('/content/TripoSR')
!find . -name "*.py" -exec sed -i 's/^import rembg/rembg = None/' {} +
os.makedirs('output/0', exist_ok=True)
!python run.py "{IMG}" --output-dir output/ --model-save-format glb --mc-resolution 320 --no-remove-bg
OUT = 'output/0/mesh.glb'
print('\nResultado:', ('OK ' + OUT + ' — ' + str(round(os.path.getsize(OUT)/1024, 1)) + ' KB | ' + time.ctime(os.path.getmtime(OUT))) if os.path.exists(OUT) else 'ERROR: revisa el error rojo')

In [ ]:
# Celda 4 — Descargar
from google.colab import files
files.download('/content/TripoSR/output/0/mesh.glb')